# Capítulo 22 - Semiose Aplicada: Camadas A/B/C no AI-Orchestrator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap22_semiose.ipynb)

Este capítulo é **prático e autoral**: reproduz, em escala reduzida e executável,
as três camadas de Semiose implementadas no **AI-Orchestrator**:

- **Camada A — Enriquecimento contextual** (determinístico; regex + sinais do turno anterior).
- **Camada A+ (S1) — Contextual Embeddings no índice** do router (prefixar exemplos com domínio antes de embedar).
- **Camada B — Knowledge Graph** (travessia dirigida no Neo4j como *tool virtual*).
- **Camada C Nível 1 — Re-ranking contextual** (boost aditivo determinístico).
- **Camada C Nível 2 (S3) — Cross-encoder como desempate** do top-2 ambíguo.
- **Avaliação (S6) — Routing Failure Rate** e *BERTScore* real.

Diferente dos capítulos 2–18 (adaptações de materiais didáticos genéricos), este
notebook **espelha o código-fonte real do projeto**. Onde a infraestrutura completa
(Qdrant/Neo4j/Ollama) seria necessária, usamos *mocks* locais para que tudo rode
em CPU/Colab sem dependências pesadas — cada célula aponta o arquivo real equivalente.

---

## Atribuição e fontes

**Origem:** material **autoral** do projeto **AI-Orchestrator** (`gateway/`), por Allan Eric Jepsen,
para o livro *LLM On-Premise: Guia Prático*.

**Arquivos reais espelhados aqui:**
- `gateway/query_enricher.py` — Camada A
- `gateway/semantic_router.py` — Camadas A+ (S1), C Nível 1 e C Nível 2 (S3)
- `gateway/knowledge_graph.py` — Camada B
- `evals/eval_semiose.py` — Avaliação (S6)
- `PLANO_SEMIOSE.md` — plano arquitetural e gates

**Fontes científicas (ver `referencias/bibliografia.md`):**
- Anthropic (2024), *Contextual Retrieval* — Contextual Embeddings/BM25.
- Reimers & Gurevych (2019), *Sentence-BERT*, arXiv:1908.10084 — bi-encoder vs cross-encoder.
- Lewis et al. (2020), *RAG*, arXiv:2005.11401 — memória não-paramétrica.
- Gao et al. (2023), *RAG Survey*, arXiv:2312.10997 — pré/pós-recuperação, rerank.
- Edge et al. (2024), *GraphRAG*, arXiv:2404.16130 — grafo de entidades.

> Todo o código deste notebook é original; trechos longos do projeto são reproduzidos
> de forma reduzida para fins didáticos, sempre indicando o arquivo equivalente.

---

## Dependências

O núcleo deste notebook (Camadas A, C-N1, S1 e a avaliação S6) roda **sem nenhuma
dependência externa** — apenas a biblioteca padrão do Python. As seções opcionais
(embeddings SBERT reais e cross-encoder real) usam `sentence-transformers`; se não
estiver instalado, o notebook cai automaticamente em *mocks* determinísticos.

Descomente a célula abaixo para habilitar as seções opcionais.

In [ ]:
# Opcional: SBERT + cross-encoder reais (CPU). Sem isso, o notebook usa mocks.
# %pip install -q sentence-transformers==3.4.1

from __future__ import annotations

try:
    from sentence_transformers import SentenceTransformer, CrossEncoder  # noqa: F401
    SBERT_AVAILABLE = True
except Exception:  # noqa: BLE001
    SBERT_AVAILABLE = False

print("sentence-transformers disponivel:", SBERT_AVAILABLE)

sentence-transformers disponivel: False


---
## Parte 1 — Camada A: Enriquecimento contextual (`gateway/query_enricher.py`)

A Camada A transforma a *query crua* em *query contextualizada* usando **apenas
metadata estruturada** do turno anterior (domínio validado + entidades extraídas
por regex). Nunca concatena texto cru do histórico — isso é uma decisão de
segurança (evita *prompt injection* via histórico).

Princípios (do projeto):
- **Harness-first**: zero chamada LLM, 100% determinístico.
- **Degradação graceful**: sem contexto → retorna a query original.
- **Detecção de troca de tópico** (`_has_strong_conflict`): se a nova pergunta
  traz palavras fortes de *outro* domínio, o enriquecimento é descartado — não
  faz sentido carimbar "domínio: vendas" numa pergunta que virou de RH.

Abaixo reproduzimos uma versão reduzida e executável da lógica real.

In [ ]:
import re
import unicodedata
from dataclasses import dataclass, field

# --- Espelho reduzido de gateway/router.py -------------------------------
def _normalize(text: str) -> str:
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text

# Subconjunto representativo das keywords reais por dominio.
_DOMAIN_KEYWORDS = {
    "financas": {"fatura", "boleto", "fluxo de caixa", "contas a pagar"},
    "rh": {"salario", "ferias", "funcionario", "cpf", "admissao"},
    "estoque": {"estoque", "sku", "inventario", "reposicao"},
    "vendas": {"comissao", "pedido", "desconto", "vendedor", "cliente"},
}

# --- Espelho de gateway/query_enricher.py --------------------------------
_SKU_RE = re.compile(r"\b[A-Z]{2,5}(?:-[A-Z0-9]{2,5})+-\d{2,5}\b")
_CPF_RE = re.compile(r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b")
_ORDER_ID_RE = re.compile(r"\b(?:pedido|PED)\s*#?\s*(\d{3,})\b", re.IGNORECASE)
_ENTITY_PATTERNS = [("sku", _SKU_RE), ("cpf", _CPF_RE), ("pedido", _ORDER_ID_RE)]
_ENTITY_DOMAIN_MAP = {"sku": "estoque", "pedido": "vendas", "cpf": "rh"}
_CROSS_DOMAIN_VOCAB = {
    "vendas": frozenset({"sku"}),
    "estoque": frozenset({"cliente"}),
    "rh": frozenset({"vendas"}),
    "financas": frozenset(),
}


@dataclass
class ContextSignal:
    last_domain: str | None = None
    recent_entities: list = field(default_factory=list)


def _regex_extract(text: str) -> list:
    out = []
    for _label, pat in _ENTITY_PATTERNS:
        out.extend(pat.findall(text))
    return out


def _has_strong_conflict(question: str, last_domain: str) -> bool:
    normalized = _normalize(question)
    cross = _CROSS_DOMAIN_VOCAB.get(last_domain, frozenset())
    for domain, kws in _DOMAIN_KEYWORDS.items():
        if domain == last_domain:
            continue
        for kw in kws:
            if kw in cross:
                continue
            if re.search(rf"(?<![a-z]){re.escape(kw)}", normalized):
                return True
    for label, pat in _ENTITY_PATTERNS:
        if pat.search(question) and _ENTITY_DOMAIN_MAP.get(label, last_domain) != last_domain:
            return True
    return False


def enrich_query(question: str, signals: ContextSignal) -> tuple[str, bool]:
    if not signals.last_domain and not signals.recent_entities:
        return question, False
    if signals.last_domain and _has_strong_conflict(question, signals.last_domain):
        return question, False
    parts = []
    if signals.last_domain:
        parts.append(f"dominio: {signals.last_domain}")
    if signals.recent_entities:
        parts.append(f"entidades: {', '.join(signals.recent_entities[:5])}")
    if not parts:
        return question, False
    return f"[{'; '.join(parts)}] {question}", True


print("Camada A carregada.")

Camada A carregada.


In [ ]:
# Demonstracao: continuidade vs troca de topico
prev = ContextSignal(last_domain="vendas", recent_entities=["CAD-ERG-001"])

casos = [
    "qual a comissao desse pedido?",          # continuidade -> enriquece
    "e o salario do vendedor Rafael?",         # 'salario' (RH) -> troca de topico
    "esse SKU ainda tem em estoque?",          # 'estoque' forte -> troca p/ estoque
]

for q in casos:
    enriched, did = enrich_query(q, prev)
    marca = "ENRIQUECIDA" if did else "original  "
    print(f"[{marca}] {q}\n            -> {enriched}\n")

[ENRIQUECIDA] qual a comissao desse pedido?
            -> [dominio: vendas; entidades: CAD-ERG-001] qual a comissao desse pedido?

[original  ] e o salario do vendedor Rafael?
            -> e o salario do vendedor Rafael?

[original  ] esse SKU ainda tem em estoque?
            -> esse SKU ainda tem em estoque?



---
## Parte 2 — Camada A+ (S1): Contextual Embeddings no índice (`semantic_router.py`)

Inspirado em **Contextual Retrieval** (Anthropic, 2024): em vez de embedar o
trecho isolado, anexamos um contexto curto que o *situa*. No router, isso significa
prefixar cada exemplo do golden com seu **domínio** antes de embedar no Qdrant.

A assimetria é intencional — igual à técnica original: o **corpus** (índice) recebe
contexto; a **query** em runtime permanece natural. O efeito é agrupar melhor os
exemplos por domínio no espaço vetorial, reduzindo falhas de recuperação.

> Recurso **opt-in** no projeto: `CONTEXTUAL_EMBEDDINGS_ENABLED=1`. Gate de validação:
> *Routing Failure Rate* antes/depois (Parte 6).

In [ ]:
# Espelho de gateway/semantic_router.py::_contextual_text
def _contextual_text(question: str, domains: list) -> str:
    if not domains:
        return question
    return f"[dominio: {', '.join(domains)}] {question}"


golden = [
    {"question": "qual a comissao do vendedor?", "domains": ["vendas"]},
    {"question": "esse SKU tem em estoque?", "domains": ["estoque"]},
    {"question": "quando sai meu salario?", "domains": ["rh"]},
]

print("Texto embedado no indice (S1 ligado):")
for r in golden:
    print(" ", _contextual_text(r["question"], r["domains"]))

Texto embedado no indice (S1 ligado):
  [dominio: vendas] qual a comissao do vendedor?
  [dominio: estoque] esse SKU tem em estoque?
  [dominio: rh] quando sai meu salario?


---
## Parte 3 — Camada C Nível 1: re-ranking contextual (`semantic_router.py`)

Após o bi-encoder recuperar os vizinhos do golden, aplicamos um **boost aditivo
determinístico** nos hits cujo domínio coincide com o `context_domain` do turno
anterior. É barato (zero LLM), preserva o `_raw_score` para *tracing* e tem *cap*
em 1.0 para manter a semântica de cosseno.

A regra de ouro: o boost **desempata matches bons**, ele *não resgata* matches ruins
(o gate de threshold continua sobre o cosseno).

In [ ]:
CONTEXT_BOOST = 0.05  # gateway/semantic_router.py::CONTEXT_BOOST


def contextual_boost(hits: list, context_domain: str | None, boost: float = CONTEXT_BOOST) -> list:
    if not context_domain:
        return hits
    for h in hits:
        h["_raw_score"] = h.get("score", 0.0)
        if context_domain in h["domains"]:
            h["score"] = min(h["_raw_score"] + boost, 1.0)
    return sorted(hits, key=lambda h: h["score"], reverse=True)


hits = [
    {"q": "salario do funcionario", "domains": ["rh"], "score": 0.90},
    {"q": "comissao do vendedor", "domains": ["vendas"], "score": 0.88},
]

print("Sem contexto:")
for h in hits:
    print(f"  {h['score']:.2f}  {h['domains']}  {h['q']}")

boosted = contextual_boost([dict(h) for h in hits], context_domain="vendas")
print("\nCom context_domain=vendas (boost +0.05):")
for h in boosted:
    raw = h.get("_raw_score", h["score"])
    print(f"  {h['score']:.2f} (raw {raw:.2f})  {h['domains']}  {h['q']}")

Sem contexto:
  0.90  ['rh']  salario do funcionario
  0.88  ['vendas']  comissao do vendedor

Com context_domain=vendas (boost +0.05):
  0.93 (raw 0.88)  ['vendas']  comissao do vendedor
  0.90 (raw 0.90)  ['rh']  salario do funcionario


---
## Parte 4 — Camada C Nível 2 (S3): cross-encoder como desempate (`semantic_router.py`)

O router usa **consenso estrito**: se o top-2 tem domínios diferentes e o *gap* de
cosseno é pequeno, a query é considerada ambígua e o router devolve `None` (deixa o
LLM classificar). Esse era o ponto onde o plano previa um "Nível 2 LLM" — nunca
implementado.

S3 substitui esse LLM por um **cross-encoder** (Reimers & Gurevych, 2019): enquanto o
bi-encoder embeda query e exemplo separadamente (rápido, bom para recuperar), o
cross-encoder processa o **par junto** (mais lento, mais preciso) e produz um score
de relevância par-a-par. Usamo-lo **só no caso ambíguo**, preservando o cosseno para
o gate de aceitação. É a integração correta para um router de consenso — um *reorder*
puro colidiria com os guards (ver nota em `PLANO_SEMIOSE.md`).

> Opt-in: `RERANK_CROSS_ENCODER_ENABLED=1`. Graceful: modelo ausente → cai no consenso → `None`.

In [ ]:
THRESHOLD, MIN_GAP = 0.80, 0.05


class FakeCrossEncoder:
    """Mock: pontua alto quando um termo-alvo aparece no exemplo."""
    def __init__(self, prefer: str):
        self.prefer = prefer
    def predict(self, pairs):
        return [1.0 if self.prefer in b else 0.0 for _q, b in pairs]


def load_cross_encoder(prefer: str):
    # Em producao seria CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1").
    # Aqui usamos o mock para rodar sem download. Troque por SBERT real se quiser.
    return FakeCrossEncoder(prefer)


def route_with_consensus(question, hits, cross_encoder=None):
    """Espelho reduzido de SemanticRouter.route + _cross_encoder_decide."""
    hits = sorted(hits, key=lambda h: h["score"], reverse=True)
    a, b = hits[0], hits[1]
    ambiguous = set(a["domains"]) != set(b["domains"]) and abs(a["score"] - b["score"]) < MIN_GAP

    if ambiguous and cross_encoder is not None:  # S3 desempata
        sa, sb = cross_encoder.predict([[question, a["q"]], [question, b["q"]]])
        winner = a if sa >= sb else b
        if winner["score"] >= THRESHOLD:
            return winner["domains"], f"desempate cross-encoder -> {winner['q']!r}"

    if ambiguous:  # consenso rejeita
        return None, "ambiguo: consenso rejeita (fallback LLM)"
    if a["score"] >= THRESHOLD:
        return a["domains"], f"consenso -> {a['q']!r}"
    return None, "abaixo do threshold"


ambiguo = [
    {"q": "salario do funcionario", "domains": ["rh"], "score": 0.95},
    {"q": "comissao do vendedor", "domains": ["vendas"], "score": 0.93},
]
pergunta = "quanto a Juliana recebeu de comissao?"

print("Sem S3:        ", route_with_consensus(pergunta, [dict(h) for h in ambiguo]))
print("Com S3 (CE):   ", route_with_consensus(pergunta, [dict(h) for h in ambiguo],
                                               cross_encoder=load_cross_encoder("comissao")))

Sem S3:         (None, 'ambiguo: consenso rejeita (fallback LLM)')
Com S3 (CE):    (['vendas'], "desempate cross-encoder -> 'comissao do vendedor'")


---
## Parte 5 — Camada B: Knowledge Graph como *tool virtual* (`knowledge_graph.py`)

Quando a resposta exige relacionar entidades de **domínios diferentes** ("quais
clientes dependem deste fornecedor em atraso?"), embeddings sozinhos não bastam —
relações são melhor representadas em **grafo** (GraphRAG, Edge et al., 2024). O
AI-Orchestrator expõe o Neo4j como uma **tool virtual**: o sub-agente "chama" uma
ferramenta que, em vez de HTTP, executa uma travessia Cypher local.

A implementação atual é **travessia dirigida** (não a sumarização hierárquica do
GraphRAG completo — ver "Honestidade arquitetural" no `PLANO_SEMIOSE.md`). Abaixo,
a query real e uma execução *mockada* (sem Neo4j no ar).

In [ ]:
# Query real de travessia (gateway/knowledge_graph.py::_EXPAND_CYPHER)
EXPAND_CYPHER = """
MATCH (e:Entity)
WHERE (e.name = $entity_name OR e.sku = $entity_name) AND e.type = $entity_type
WITH e
MATCH (e)-[r*1..2]-(related:Entity)
WHERE related <> e AND ($target_domain = '' OR related.domain = $target_domain)
WITH DISTINCT related, [rel IN r | type(rel)] AS path_types
RETURN related.name AS name, related.type AS type,
       related.domain AS domain, path_types
ORDER BY related.domain, related.name
LIMIT $limit
"""
print(EXPAND_CYPHER)


def expand_mock(entity_name, entity_type, target_domain=""):
    """Simula KnowledgeGraph.expand sem Neo4j. Em producao, roda o Cypher acima."""
    grafo = {
        ("FORN-042", "fornecedor"): [
            {"name": "CAD-ERG-001", "type": "produto", "domain": "estoque", "path_types": ["FORNECE"]},
            {"name": "Cliente ACME", "type": "cliente", "domain": "vendas", "path_types": ["FORNECE", "COMPRADO_POR"]},
        ],
    }
    res = grafo.get((entity_name, entity_type), [])
    if target_domain:
        res = [r for r in res if r["domain"] == target_domain]
    return res


print("\nExpansao de FORN-042 (fornecedor) -> dominio vendas:")
for r in expand_mock("FORN-042", "fornecedor", target_domain="vendas"):
    print("  ", r)


MATCH (e:Entity)
WHERE (e.name = $entity_name OR e.sku = $entity_name) AND e.type = $entity_type
WITH e
MATCH (e)-[r*1..2]-(related:Entity)
WHERE related <> e AND ($target_domain = '' OR related.domain = $target_domain)
WITH DISTINCT related, [rel IN r | type(rel)] AS path_types
RETURN related.name AS name, related.type AS type,
       related.domain AS domain, path_types
ORDER BY related.domain, related.name
LIMIT $limit


Expansao de FORN-042 (fornecedor) -> dominio vendas:
   {'name': 'Cliente ACME', 'type': 'cliente', 'domain': 'vendas', 'path_types': ['FORNECE', 'COMPRADO_POR']}


---
## Parte 6 — Avaliação (S6): Routing Failure Rate e BERTScore (`evals/eval_semiose.py`)

Toda melhoria precisa de um **gate mensurável**. S6 adiciona duas instrumentações:

1. **Routing Failure Rate** — fração de casos em que o conjunto de domínios previsto
   difere do esperado (conjunto vazio/None conta como falha). Espelha a métrica de
   *failed retrievals* da Anthropic (−49% com Contextual Embeddings; −67% com rerank).
2. **BERTScore real** (token-level, via pacote `bert-score`) para preservação semântica
   do enriquecimento — mais fiel que o proxy de cosseno usado por padrão. É **opcional**:
   se a lib não estiver instalada, a métrica fica em 0.

Abaixo, a função real e uma comparação *antes/depois* simulada de S1+S3.

In [ ]:
# Funcao real: evals/eval_semiose.py::routing_failure_rate
def routing_failure_rate(predicted: list, expected: list) -> float:
    if not expected:
        return 0.0
    failures = sum(1 for p, e in zip(predicted, expected) if p != e)
    return failures / len(expected)


# Golden minimo: esperado vs previsto antes (sem S1/S3) e depois (com S1/S3).
expected = [{"vendas"}, {"rh"}, {"estoque"}, {"vendas"}]
antes    = [{"vendas"}, set(),  {"estoque"}, set()]      # 2 falhas (None em ambiguos)
depois   = [{"vendas"}, {"rh"}, {"estoque"}, {"vendas"}]  # S3 desempata os ambiguos

rfr_antes = routing_failure_rate(antes, expected)
rfr_depois = routing_failure_rate(depois, expected)
print(f"Routing Failure Rate  antes (sem S1/S3): {rfr_antes:.2%}")
print(f"Routing Failure Rate depois (com S1/S3): {rfr_depois:.2%}")
print(f"Reducao relativa: {(rfr_antes - rfr_depois) / rfr_antes:.0%}" if rfr_antes else "n/a")

Routing Failure Rate  antes (sem S1/S3): 50.00%
Routing Failure Rate depois (com S1/S3): 0.00%
Reducao relativa: 100%


In [ ]:
# BERTScore real (opcional) - evals/eval_semiose.py::_try_bertscore
def try_bertscore(refs: list, hyps: list):
    if not refs or not hyps:
        return None
    try:
        from bert_score import score as _bs
    except Exception:
        return None
    _p, _r, f1 = _bs(hyps, refs, lang="pt", verbose=False)
    return float(f1.mean())


refs = ["qual a comissao desse pedido?"]
hyps = ["[dominio: vendas; entidades: CAD-ERG-001] qual a comissao desse pedido?"]
bs = try_bertscore(refs, hyps)
print("BERTScore F1:", f"{bs:.4f}" if bs is not None else "bert-score nao instalado (metrica = 0)")

BERTScore F1: bert-score nao instalado (metrica = 0)


---
## Conclusão e mapa para o código real

Reproduzimos, de forma executável, o pipeline de Semiose do AI-Orchestrator:

| Camada | Conceito | Arquivo real | Flag |
|--------|----------|--------------|------|
| A | Enriquecimento determinístico | `gateway/query_enricher.py` | `ENRICHER_ENABLED` |
| A+ (S1) | Contextual Embeddings no índice | `gateway/semantic_router.py::_contextual_text` | `CONTEXTUAL_EMBEDDINGS_ENABLED` |
| C-N1 | Boost contextual | `gateway/semantic_router.py` | `RERANK_ENABLED` |
| C-N2 (S3) | Cross-encoder desempate | `gateway/semantic_router.py::_cross_encoder_decide` | `RERANK_CROSS_ENCODER_ENABLED` |
| B | Knowledge Graph (travessia) | `gateway/knowledge_graph.py` | `NEO4J_ENABLED` |
| Eval (S6) | Routing Failure Rate + BERTScore | `evals/eval_semiose.py` | — |

**Princípios transversais:** Harness antes de Model (determinístico primeiro),
*opt-in* por flag, e degradação graceful (toda dependência ausente vira fallback,
nunca erro). O texto conceitual completo está em `livro/cap22_semiose_aplicada.md`;
o plano e os gates de validação em `PLANO_SEMIOSE.md`.

### Referências
- Anthropic (2024). *Introducing Contextual Retrieval*.
- Reimers, N. & Gurevych, I. (2019). *Sentence-BERT*. arXiv:1908.10084.
- Lewis, P. et al. (2020). *Retrieval-Augmented Generation*. arXiv:2005.11401.
- Gao, Y. et al. (2023). *Retrieval-Augmented Generation for LLMs: A Survey*. arXiv:2312.10997.
- Edge, D. et al. (2024). *From Local to Global: A Graph RAG Approach*. arXiv:2404.16130.